In [1]:
import os
from anthropic import Anthropic

# Test simple API call

In [3]:

client = Anthropic(
    # This is the default and can be omitted
    api_key=os.environ.get("API_KEY"),
)

message = client.messages.create(
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": "Hello, Claude",
        }
    ],
    model="claude-opus-4-7",
)
print(message.content)

[TextBlock(citations=None, text='Hello! How can I help you today?', type='text')]


# Use claude API to annotate scraped hand images

In [4]:
import glob
import base64
from pathlib import Path
import pandas as pd
import time
from typing import Optional, List, Dict

In [14]:
def get_image_files(folder_path: str) -> List[str]:
    """
    Get all image files from a folder.
    
    Args:
        folder_path: Path to the folder containing images
    
    Returns:
        List of image file paths (supports jpg, jpeg, png, gif, webp)
    """
    image_extensions = ('*.jpg', '*.jpeg', '*.png', '*.gif', '*.webp')
    image_files = []
    
    for ext in image_extensions:
        pattern = os.path.join(folder_path, ext)
        image_files.extend(glob.glob(pattern, recursive=False))
    
    # Remove duplicates while preserving order
    seen = set()
    unique_files = []
    for f in sorted(image_files):
        if f.lower() not in seen:
            seen.add(f.lower())
            unique_files.append(f)
    
    return unique_files


def encode_image_to_base64(image_path: str) -> str:
    """
    Encode an image file to base64 string.
    
    Args:
        image_path: Path to the image file
    
    Returns:
        Base64 encoded string of the image
    """
    with open(image_path, 'rb') as image_file:
        return base64.standard_b64encode(image_file.read()).decode('utf-8')


def get_image_media_type(image_path: str) -> str:
    """
    Get the media type of an image based on file extension.
    
    Args:
        image_path: Path to the image file
    
    Returns:
        Media type string (e.g., 'image/jpeg')
    """
    ext = Path(image_path).suffix.lower()
    media_types = {
        '.jpg': 'image/jpeg',
        '.jpeg': 'image/jpeg',
        '.png': 'image/png',
        '.gif': 'image/gif',
        '.webp': 'image/webp'
    }
    return media_types.get(ext, 'image/jpeg')

In [15]:
def annotate_hand_image(image_path: str, client: Anthropic) -> Optional[str]:
    """
    Annotate a hand image using Claude to describe hand shape characteristics.
    
    Args:
        image_path: Path to the image file
        client: Anthropic client instance
    
    Returns:
        Text annotation describing hand shape characteristics, or None if failed
    """
    try:
        # Encode image to base64
        image_data = encode_image_to_base64(image_path)
        media_type = get_image_media_type(image_path)
        
        # Call Claude API with vision capability
        message = client.messages.create(
            model="claude-opus-4-7",
            max_tokens=300,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": media_type,
                                "data": image_data
                            }
                        },
                        {
                            "type": "text",
                            "text": "Provide a brief, scientific description of the hand shape characteristics in this image. Use concise language similar to biological taxonomic descriptions. Include: hand orientation, finger positioning (flexion/extension state), palm orientation, and distinctive morphological features. Avoid subjective gesture interpretation. Format as 1-2 sentences maximum."
                        }
                    ]
                }
            ]
        )
        
        # Extract the text annotation
        annotation = message.content[0].text
        return annotation
    
    except Exception as e:
        print(f"Error annotating {image_path}: {e}")
        return None

In [16]:
def annotate_all_hand_images(images_folder: str, client: Anthropic) -> List[Dict]:
    """
    Annotate all hand images in a folder and return results.
    
    Args:
        images_folder: Path to folder containing hand images
        client: Anthropic client instance
    
    Returns:
        List of dictionaries with filename and annotation
    """
    image_files = get_image_files(images_folder)
    
    if not image_files:
        print(f"No images found in {images_folder}")
        return []
    
    print(f"Found {len(image_files)} images. Starting annotation process...")
    annotations = []
    
    for idx, image_path in enumerate(image_files):
        filename = os.path.basename(image_path)
        print(f"\n[{idx + 1}/{len(image_files)}] Annotating: {filename}")
        
        annotation = annotate_hand_image(image_path, client)
        
        if annotation:
            annotations.append({
                'filename': filename,
                'annotation': annotation
            })
            print(f"  ✓ Annotation complete")
        else:
            print(f"  ✗ Failed to annotate")
        
        # Add a small delay to be respectful to the API
        time.sleep(0.5)
    
    print(f"\n✓ Total images annotated: {len(annotations)}/{len(image_files)}")
    return annotations

In [17]:
# Annotate all hand images
images_folder = os.path.join('scrapped_data', 'hand_images')

# Check if folder exists
if not os.path.exists(images_folder):
    print(f"Error: Folder not found: {images_folder}")
else:
    # Run annotation process
    hand_annotations = annotate_all_hand_images(images_folder, client)
    
    # Convert to DataFrame
    df_annotations = pd.DataFrame(hand_annotations)
    
    if len(df_annotations) > 0:
        print("\n" + "="*60)
        print("Hand Image Annotations")
        print("="*60)
        print(df_annotations)
        
        # Save to CSV
        output_dir = 'scrapped_data'
        os.makedirs(output_dir, exist_ok=True)
        annotations_output_path = os.path.join(output_dir, 'hand_image_annotations.csv')
        df_annotations.to_csv(annotations_output_path, index=False)
        print(f"\n✓ Annotations saved to: {annotations_output_path}")
    else:
        print("No annotations were created.")

Found 281 images. Starting annotation process...

[1/281] Annotating: hand_image_0001.jpg
  ✓ Annotation complete

[2/281] Annotating: hand_image_0002.jpg
Error annotating scrapped_data\hand_images\hand_image_0002.jpg: list index out of range
  ✗ Failed to annotate

[3/281] Annotating: hand_image_0003.jpg
  ✓ Annotation complete

[4/281] Annotating: hand_image_0004.jpg
  ✓ Annotation complete

[5/281] Annotating: hand_image_0005.jpg
  ✓ Annotation complete

[6/281] Annotating: hand_image_0006.jpg
  ✓ Annotation complete

[7/281] Annotating: hand_image_0007.jpg
  ✓ Annotation complete

[8/281] Annotating: hand_image_0008.jpg
  ✓ Annotation complete

[9/281] Annotating: hand_image_0009.jpg
  ✓ Annotation complete

[10/281] Annotating: hand_image_0010.jpg
  ✓ Annotation complete

[11/281] Annotating: hand_image_0011.jpg
  ✓ Annotation complete

[12/281] Annotating: hand_image_0012.jpg
  ✓ Annotation complete

[13/281] Annotating: hand_image_0013.jpg
  ✓ Annotation complete

[14/281] Annot